In [ ]:
%%html
<style>
    h1 {color:purple}
    h2 {color:purple}
    h3 {color:#0099ff}
    hr {    
        border: 0;
        height: 3px;
        background: #333;
        background-image: linear-gradient(to right, limegreen, deepskyblue, limegreen);
    }
</style>

# Outline
0. Part 0 — Intro/Setup
1. Part 1 — OpenAI Python APIs
2. **Part 2 — OpenAI Agents SDK**
   * 02-00. Agents SDK Introduction: building blocks, ReAct pattern
   * 02-01. Single Agent System — Python Tutor: `Agent`, `Runner.run()`, `RunResult`
   * 02-02. Conversation State in Agents: `previous_response_id`, `SQLiteSession`, `conversation_id`, `result.to_input_list()`
   * 02-03. Streaming Text and Events: `Runner.run_streamed()`, `StreamEvent`, `run_item_stream_event`
   * 02-04. **Python Tutor with a Model-Backed Input Guardrail: `@input_guardrail`, `GuardrailFunctionOutput`, `InputGuardrailTripwireTriggered`**
   * 02-05. Tools
      * 02-05-00. Tools Overview: hosted, local, and custom tool taxonomy
      * 02-05-01. Financial Research Agent with a Custom Tool: `@function_tool`, hosted and custom tools
      * 02-05-02. Image Generation and Editing with `ImageGenerationTool`
      * 02-05-03. Multi-Agent Deitel Book Concierge: `FileSearchTool`, RAG, vector stores, handoffs
      * 02-05-04. Code Interpreter Tool: `CodeInterpreterTool`, hosted container
      * 02-05-05. Hosted MCP — Weather and Geocoding: `MCPServerStreamableHttp`, MCP authentication
      * 02-05-06. Local MCP — SQLite Database: `MCPServerStdio`
      * 02-05-07. AccuWeather Agent with `ComputerTool`: `AsyncComputer`, Playwright, `LocalBrowserComputer`
      * 02-05-08. `ShellTool` Folder Inspector: custom executor, human-in-the-loop approval
      * 02-05-09. Local LLM via LiteLLM and Ollama: `LitellmModel`
      * 02-05-10. Python Code Tutor with Dynamic Instructions: callable instructions, `RunContextWrapper`, typed context
3. Part 3 — Codex App
4. Part 4 — Wrap-Up and Additional References

---


# Creating Agents with the OpenAI Agents SDK
---


## Guardrails Overview
* Guardrails validate inputs/outputs

### Input guardrails
* Inspect user input before (in sequence) or alongside (in parallel) the main agent run
* Executed only for the first agent in a multiagent app

### Output guardrails
* Inspect final agent response before it reaches the user
    * Detect/block personal data 
    * Ensure response has correct format (JSON, Markdown, ...)
    * Block disallowed claims (e.g., don't provide legal, medical or financial advice)
    * Confirm response includes required source citations
    * Prevent agent from revealing internal instructions, hidden notes, or private context

### Input/Output Tool guardrails
* Can validate custom function-tool inputs and outputs

### Guardrail Outputs
* Guardrail functions return `GuardrailFunctionOutput` 
    * `output_info` — details you can log or show when blocked
    * `tripwire_triggered` — `True` stops the run and raises an exception
* This example uses a **model-backed classifier agent** so the guardrail can reason about intent 
---


## Python Tutor with a Model-Backed Input Guardrail
* Starts with the same Python Tutor from `02-01`
* Adds one model-backed input guardrail that blocks off-topic prompts
* Demonstrates `InputGuardrail`, `GuardrailFunctionOutput`, and `InputGuardrailTripwireTriggered`
* Uses a small classifier agent with a structured Pydantic output
* Uses `run_in_parallel=False` so the guardrail completes before the main tutor agent starts


## Imports

In [ ]:
from agents import (
    Agent,
    Runner,
    trace,
    GuardrailFunctionOutput, 
    InputGuardrailTripwireTriggered, 
    RunContextWrapper,
    input_guardrail
)
from IPython.display import Markdown, display
from pydantic import BaseModel

## Defining the Guardrail Agent 

### Pydantic Subclass for Guardrail Agent's Return Type

In [ ]:
class TopicCheck(BaseModel):
    """Structured output from the guardrail classifier agent."""
    is_python_question: bool
    reason: str

### LLM-Backed Guardrail Agent to Check Whether a Topic is Python-Related

In [ ]:
topic_checker = Agent(
    name='Python Topic Checker',
    model='gpt-5.5',
    instructions="""Decide whether the user's request is about Python or
        introductory programming. Return false for unrelated topics.""",
    output_type=TopicCheck
)

### Input Guardrail Function Launches the Guardrail Agent
* `RunContextWrapper` parameter contains the main agent's context and is not used for this example
* Could contain items like the following that guardrail uses to determine whether agent's input is allowed
    * e.g., authentication info, permissions, ...

In [ ]:
@input_guardrail(run_in_parallel=False) # run_in_parallel=True is the default 
async def python_topic_guardrail(
    ctx: RunContextWrapper, # main agent's context provided by the SDK
    agent: Agent, # the agent this guardrail applies to
    input_data: str) -> GuardrailFunctionOutput:
    
    """Use a classifier agent to block non-Python/non-programming prompts."""
    result = await Runner.run(topic_checker, input_data, context=ctx.context)
    check = result.final_output

    return GuardrailFunctionOutput(
        output_info=check,
        tripwire_triggered=not check.is_python_question
    )

## Tutor Agent with Input Guardrail Specified

In [ ]:
tutor = Agent(
    name='Python Tutor',
    model='gpt-5.5',
    instructions="""You are a Python tutor for novice programmers.
        Provide concise, simple answers to Python questions.""",
    input_guardrails=[python_topic_guardrail]
)

In [ ]:
print('I am a Python tutor. Ask me a Python question, or type exit to quit.')
turn_count = 1

with trace(f'02-04-python-tutor-input-guardrail-[{turn_count}]'):
    while ((prompt := input(f'\nInput [{turn_count}]: ')) != 'exit'):
        try:
            result = await Runner.run(tutor, prompt)
            display(Markdown(result.final_output))
        except InputGuardrailTripwireTriggered as exc:
            check = exc.guardrail_result.output.output_info
            display(Markdown(
                '**Blocked by input guardrail.**\n\n'
                f'{check.reason}'
            ))
    
        turn_count += 1

print('User terminated app.')


## Sample Inputs
These should pass:
* What are the `print` function's parameters?
* How do I loop through a list?
* What is a Python dictionary?

These should be blocked:
* What is the best recipe for chocolate chip cookies?
* What's the weather in Boston, MA?
---

## Learn More
* [Agents](https://openai.github.io/openai-agents-python/agents/)
* [Running agents](https://openai.github.io/openai-agents-python/running_agents/)
* [Results](https://openai.github.io/openai-agents-python/results/)
* [Guardrails](https://openai.github.io/openai-agents-python/guardrails/)
* [Guardrails API reference](https://openai.github.io/openai-agents-python/ref/guardrail/)

---
© 2026 by Deitel & Associates, Inc. All Rights Reserved.